<h1>Chapter 3 - Reasoning LLMs</h1>
<i>Exploring Reasoning</i>


<a href="https://www.amazon.com/Illustrated-Guide-AI-Agents-Concepts/dp/B0GTYL2QSJ"><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="https://www.oreilly.com/library/view/an-illustrated-guide/9798341662681/"><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="https://github.com/HandsOnLLM/An-Illustrated-Guide-To-AI-Agents"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HandsOnLLM/An-Illustrated-Guide-To-AI-Agents/blob/main/chapter03/chapter03.ipynb)

---

This notebook is for Chapter 3 of [An Illustrated Guide to AI Agents](https://www.amazon.com/Illustrated-Guide-AI-Agents-Concepts/dp/B0GTYL2QSJ) by [Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/) and [Jay Alammar](https://www.linkedin.com/in/jalammar).

---

<a href="https://www.amazon.com/Illustrated-Guide-AI-Agents-Concepts/dp/B0GTYL2QSJ">
<img src="https://learning.oreilly.com/covers/urn:orm:book:9798341662681/400w/" width="350"/></a>


### **[OPTIONAL]** - Installing Packages on Google Colab <img src="https://upload.wikimedia.org/wikipedia/commons/d/d0/Google_Colaboratory_SVG_Logo.svg" width=100>

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** one of the following codeblock to install the dependencies for this chapter. If you want to use a cloud provider, you only need to run the following code block:

In [ ]:
# %%capture
# !pip install illustrated-agents

---

💡 **NOTE**: If you want to use the GPU with `ollama`, then you will have to select a GPU first. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**. 

Then, **uncomment** and run this codeblock:

---

In [ ]:
# !apt-get install -y zstd > /dev/null 2>&1 && curl -fsSL https://ollama.com/install.sh | sh
# !nohup ollama serve > /dev/null 2>&1 & sleep 3 && ollama pull gemma4:e4b && ollama pull gemma3:12b &

## Choosing Your LLM

At the beginning of every chapter, we start by choosing the LLM that we want to use:

In [ ]:
from illustrated_agents.chapters.ch2 import LLM

# Gemma 3 12B (no native thinking or tool calling)
llm = LLM(model="gemma3:12b", temperature=1)

If you want to use another LLM, here are a couple of options (both locally and on the cloud) that you can try:

In [ ]:
# # llama.cpp
# llm = LLM(model="gemma-3-12B-it-Q4_K_M", base_url="http://127.0.0.1:8080", temperature=1)

# # LM Studio
# llm = LLM(model="gemma-3-12b-it", base_url="http://127.0.0.1:1234/v1", temperature=1)

# # OpenRouter
# import os
# llm = LLM(model="google/gemma-3-12b-it", base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"], temperature=1)

Note that we set the `temperature` to 1 as that will allow us to generate different answers that we can use later on for sampling. If we were to set it to 0, then the output would always be the same regardless of how many times we run it.
After doing so, let’s give the LLM a riddle and ask it to be concise in its answer as an example. As discussed, this is a form of few-shot learning where you give examples of what you want the answer the interaction be.


<hr style="height: 5px; border: none; border-radius: 5px; background: linear-gradient(to right, #000000, #7D7D7D);" />

## 1 - Chain-of-Thought

Throughout this chapter, we covered various ways that LLMs could be made to showcase reasoning. Although much of the chapter focuses on adding reasoning through training, we will show you how to add reasoning through prompting instead. Training can take quite a while and requires significant compute (many GPUs), so showcasing the foundations through prompting helps give you a basic overview of how it works.

In this notebook, we will be showing various ways to add *reasoning* to a non-reasoning model. Note that we are not directly going to use this in your `TinyAgent` since adding reasoning will be explored in more detail in Chapter 6. In this chapter, we merely explore how you can enable reasoning through prompting and fine-tuning.


![../images/ch3.png](../images/ch3.png)

Before we explore chain-of-thought in more detail, let's start with a basic example one-shot/few-shot learning where you provide one or more examples to the LLM of user/assistant roles demonstrating the behavior you'd want to see. 

First, imagine if the LLM has seen various examples where it would generate a short answer. Upon seeing that, the LLM typically decides to reproduce that behavior:

In [ ]:
from rich import print

# Examples of few-shot prompting
prompt_example = "I saw 6 flamingos. 2 flew away. 1 hid behind a tree. How many can I see? Be concise."
prompt_answer = "3"

# The query we want to ask the model
query = "I saw 9 penquins. 2 slid in the water and disappeared from sight whilst 4 waddled up from the shore. How many can I see?"

# Messages
messages = [
    {"role": "user", "content": prompt_example},
    {"role": "assistant", "content": prompt_answer},
    {"role": "user", "content": query},
]

# Generate response
response = llm.generate(messages)
print(response)

However, the answer is incorrect. It only gives back the answer directly without any reasoning. Using Chain-of-Thought, we can enable this behavior by showing the model how it should answer. The `prompt_answer` shows how we would the model to reason. By using this example, the model tends to output reasoning first before giving the final answer:

In [ ]:
prompt_example = "I saw 6 flamingos. 2 flew away. 1 hid behind a tree. How many can I see?"
prompt_answer = "You saw 6 flamingos 2 flew away. 6 - 2 = 4. Then, 1 flamingo hid. You see now 4 - 1 = 3."

# Messages
messages = [
    {"role": "user", "content": prompt_example},
    {"role": "assistant", "content": prompt_answer},
    {"role": "user", "content": query},
]

# Generate response
response = llm.generate(messages)
print(response)

This time the answer is correct! Notice how the model decides to reason before giving an answer. 

## 2 - Zero-shot Chain-of-Thought

Using examples wastes a lot of tokens and often requires having examples that actually matches the query, which isn't always the case. Instead, we can create reasoning behavior by using the additional prompt "Let's think step by step". More recent models have seen more reasoning data during training and can be nudged to reasoning more extensively by simply asking for it:

In [ ]:
# Messages with zero-shot Chain-of-Thought prompting
messages = [{"role": "user", "content": query + " Let's think step by step."}]

# Generate response
response = llm.generate(messages)
print(response)

We again see longer reasoning and a correct answer!

In the `TinyAgent`, we can use the same form of zero-shot Chain-of-Thought to create reasoning behavior:

In [ ]:
from illustrated_agents.chapters.ch2 import TinyAgent

agent = TinyAgent(llm=llm)
response = agent.run(query + " Let's think step by step.")

In [ ]:
print(response)

## 3 - Self-Consistency

Self-consistency samples a user-defined number of answers and performs a majority vote to select the most frequent answer. 
To generate many different reasoning traces and answers, a high temperature is usually combined with Chain-of-Thought-like prompting. 

Let’s explore how we would do this in practice. We use a seating puzzle that the model has to reason through. 


The model is queried 10 times and can each time generate a different reasoning trace and answer. We ask the model to put its answer after “Answer:” so we can easily separate the answer from the reasoning. The answers are then counted:

> **NOTE**: Running the model 10 times can take quite a while depending on your hardware, so feel free to skip this if it takes too long.

In [ ]:
from collections import Counter

# Self-consistency needs sampling diversity; we explicitly use temperature=1.
llm = LLM(model="gemma3:12b", temperature=1)

answers = []
for _ in range(10):
    query = """Six friends (Maarten, Ilse, Sarah, Jor, Irene, and Chris) are sitting in a row.

- Sarah is in seat 3.
- Chris is sitting at one of the ends of the row.
- Jor is sitting immediately to the right of Chris.
- Ilse is sitting exactly in the middle of Sarah and Maarten.
- Irene is not sitting next to Sarah.
- Maarten is sitting somewhere to the left of Irene.

In which seat is Maarten sitting?
Let's think step by step and give back your answer only after 'Answer:'.
"""

    # Generate response
    messages = [{"role": "user", "content": query},]
    response = llm.generate(messages)
    answer = response.content.split("Answer:")[-1].replace("\n", "").strip()
    answers.append(answer)

Counter(answers)

The code example above illustrates this nicely as it is right on the edge of what the model knows. In our experiments, the model tends to lean towards the correct answer (5) but sometimes might produce too many incorrect answers. Since we have a temperature of 1 the results will be different each time you run the model. As a result, don't be surprised if the model tends to get the answer wrong, that shows how much on the edge this question is for the capabilities of this model. 

In our tests, it tends to doubt between Seat 1 and Seat 5 which shows you that even though self-consistency might help, it is by no means a perfect technique.

Try it out with one of your own queries and see if self-consistency helps! As you might notice, it isn't the most stable technique and can be quite costly if the model has a low chance of getting the answer right. 
 

## 4 - Best-of-N

A common and straightforward method for using a verifier is called Best-of-N samples. In this method, the LLM generates N candidate answers, typically using a high or varying temperature to encourage diversity. Then, a verifier evaluates each answer and selects the highest-scoring one.

Let’s go through an example on how we could verify the output of an LLM and choose the best one through sampling. We will ask the model to generate a function that converts Roman numerals (e.g., IV) to integers (e.g., 4). Let’s assume that we have a couple of test cases. With those, we can create a simple verification function that checks how many test cases pass:


In [ ]:
test_cases = [
    ("III", 3),
    ("IV", 4),
    ("IX", 9),
    ("LVIII", 58),
    ("MCMXCIV", 1994),
    ("MMMCMXCIX", 3999),
    ("", 0),
    ("IIII", 0),       # invalid: four in a row
    ("VV", 0),         # invalid: V can't repeat
    ("IC", 0),         # invalid: I can only subtract from V or X
    ("ABC", 0),        # invalid: non-Roman characters
    ("MMMM", 0),       # invalid: exceeds 3999
]


def verify(answer):
    try:
        # Extract the function
        namespace = {}
        exec(answer, namespace)
        fn = namespace["roman_to_int"]

        # Evaluate the function on test cases
        score = sum(fn(inp) == expected for inp, expected in test_cases) / len(test_cases)
        return score
    except Exception:
        # Return 0 if the code is not valid Python
        return 0.0


Using those test cases, we can run a similar loop as we did with the self-consistency example. 

> **NOTE**: Running the model 10 times can take quite a while depending on your hardware, so feel free to skip this if it takes too long.

In [ ]:
responses = []
for _ in range(10):
    # The query we want to ask the model
    query = """Write a Python function `roman_to_int(s)` that converts a Roman numeral 
string to an integer. The function should:
- Handle standard Roman numerals from 1 to 3999.
- Correctly handle subtractive notation (e.g., IV=4, IX=9, XL=40, XC=90, CD=400, CM=900).
- Return 0 for invalid inputs (empty string, non-Roman characters, or invalid patterns 
  like "IIII" or "VV").

Give only the function definition, no explanation or markdown formatting."""

    # Generate response
    response = llm.generate([{"role": "user", "content": query}])

    # Verify the response and compute a score
    score = verify(response.content)

    # Store the response and its score
    responses.append((response.content, score))

# Get highest score and the corresponding answer
best_answer, best_score = max(responses, key=lambda x: x[1])

# Show all scores
[response[1] for response in responses]

Note that we have found only 1 answer that is correct. Without having a verify function and without sampling, it is unlikely the model would have gotten the correct answer.

## 5 - Native Reasoning

In the previous chapter, we already covered much of native reasoning. Many LLMs "just" do without any prompting! That is because, as covered in the book, these models are typically trained to perform any reasoning between special tokens. The model that we are using, for instance, has the following prompt template:

<pre style="background:#1e1e1e; color:#d4d4d4; padding:16px; border-radius:8px; overflow-x:auto; font-size:13px; line-height:1.8;">
<span style="color:#f97583;">&lt;|turn&gt;system</span>
<span style="color:#4DDC52;">&lt;|think|&gt;</span><span style="background:#D7D7D7; color:#000000; padding:1px 6px; border-radius:3px;">SYSTEM_PROMPT</span><span style="color:#f97583;">&lt;turn|&gt;</span>
<span style="color:#f97583;">&lt;|turn&gt;user</span>
<span style="background:#D7D7D7; color:#000000; padding:1px 6px; border-radius:3px;">USER_PROMPT</span><span style="color:#f97583;">&lt;turn|&gt;</span>
<span style="color:#f97583;">&lt;|turn&gt;model</span>
</pre>

Let's use the Gemma 4 E4B model that supports this template:

In [ ]:
# Ollama through OpenAI API
llm = LLM(model="gemma4:e2b", think=True)

Under the hood, LLM inferencing engines do the prompt templating themselves. So all you have to do is tell it (ollama in this example) that you want thinking behavior enabled, and it will add the `<|think|>` token for you.



In [ ]:
# The query we want to ask the model
query = "I saw 9 penquins. 2 slid in the water and disappeared from sight whilst 4 waddled up from the shore. How many can I see?"

# Generate response
response = llm.generate([{"role": "user", "content": query}])
print(response)

See how `reasoning` is now automatically extracted? In the early days of LLMs, you had to be careful with the prompt template and make sure you did it exactly as was shown by the authors. Now, this is fortunately being handled by the provider because frankly, it was quite annoying to do it yourself!

<hr style="height: 5px; border: none; border-radius: 5px; background: linear-gradient(to right, #000000, #7D7D7D);" />

# What We Built

Nothing!

At least, not something that needs to be added to the `TinyAgent`. The reasoning behavior of Gemma 3 12B was "added" by simply modifying the prompt. We also already had the `think` parameter built in Chapter 2 to tell the inference provider (Ollama) that we wanted the model to use the `<|think|>` token. 

Instead, we explored in this chapter what it means to perform reasoning through prompting and how you can use Gemma 4 E2B to perform native reasoning.

# What's Next

In the next chapter, we will explore how to further enhance your `TinyAgent` with memory!